<a href="https://colab.research.google.com/github/Sageh9/MSSP607/blob/main/week9_MSSP6070.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import os
import requests

def fetch_image_pages(term, limit):
    url = "https://commons.wikimedia.org/w/api.php"
    headers = {"User-Agent": "StudentProjectBot/1.0 (https://example.com)"}
    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": term,
        "gsrlimit": str(limit),
        "gsrnamespace": "6",
        "prop": "imageinfo",
        "iiprop": "url",
        "format": "json"
    }
    r = requests.get(url, params=params, headers=headers, timeout=20)
    r.raise_for_status()
    data = r.json()
    pages = data.get("query", {}).get("pages", {})
    results = []
    for pid, p in pages.items():
        iinfo = p.get("imageinfo")
        if not iinfo:
            continue
        imgurl = iinfo[0].get("url")
        title = p.get("title")
        if imgurl:
            results.append((title, imgurl))
    return results

def download_images(results, outdir):
    os.makedirs(outdir, exist_ok=True)
    for idx, (title, url) in enumerate(results, 1):
        ext = os.path.splitext(url.split("?")[0])[1]
        if not ext:
            ext = ".jpg"
        fname = f"{idx:03d}_{title.replace('File:','').replace('/','_')}{ext}"
        safe_name = "".join(c for c in fname if c.isalnum() or c in "._- ")
        path = os.path.join(outdir, safe_name)
        headers = {
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
            "Referer": "https://commons.wikimedia.org/"
        }
        try:
            with requests.get(url, headers=headers, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(path, "wb") as f:
                    for chunk in r.iter_content(8192):
                        f.write(chunk)
            print(f"Saved {path}")
        except Exception as e:
            print(f"Failed {url} -> {e}")


def main():
    term = input("Enter a keyword to search for images: ")
    limit = input("How many images to download? (default 10): ")
    if not limit.strip():
        limit = 10
    else:
        limit = int(limit)
    results = fetch_image_pages(term, limit)
    if not results:
        print("No images found.")
        return
    outdir = f"images_{term.replace(' ','_')}"
    download_images(results, outdir)
    print("Done.")

if __name__ == "__main__":
    main()


Enter a keyword to search for images: cat
How many images to download? (default 10): 10
Saved images_cat/001_20231125 housecat south meadows PD100306.jpg.jpg
Saved images_cat/002_Cat November 2010-1a.jpg.jpg
Saved images_cat/003_Cat Sphynx. Kittens. img 11.jpg.jpg
Saved images_cat/004_Cat playing with a lizard.jpg.jpg
Saved images_cat/005_Felis catus-cat on snow.jpg.jpg
Saved images_cat/006_June odd-eyed-cat cropped.jpg.jpg
Saved images_cat/007_Six weeks old cat aka.jpg.jpg
Saved images_cat/008_Sleeping cat on her back.jpg.jpg
Saved images_cat/009_Tabby cat with blue eyes-3336579.jpg.jpg
Saved images_cat/010_Tired 20-year-old cat.jpg.jpg
Done.
